This is the notebook to define the 8 Qubit QAOA in Bloqade SDK using python:

We need Hc, which is the 8 values that come out of H function after passing it the feateures variables

We need Hm, which we define it, a common way to define it is using RX gate (rotations in X)

We need to define the QAOA algorithm, that in essence is: 
- Starting the 8 qubits
- Apply Hadamard gates to create equal superposition
- Apply Hc part of the circuit which would be quantum gates 
- Apply Hm part of the circuit which would be quantum gates
- Measure it   

** After measurement you get a bitstring **

After having your QAOA defined, we also have to define auxiliar functions in python to make x quantity of shots per circuit, define the probability of a bitstring, the function to calculate energy/cost of a bitstring, a classical optimizer to update parameters from QAOA, run the circuit again, so we need the function to run the circuit

**pip installs**

In [5]:
pip install -q bloqade 

Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys
!{sys.executable} -m pip install -U lark

**Imports**

In [30]:
import math
from typing import Any
import numpy as np 
import kirin
from kirin.dialects import ilist

from bloqade import qasm2

pi = math.pi

**Global variables from Hamiltonian function we produced**

With these variables produced already, we do not need to use the dataset again or touch it in this ipynb because we already produced the hamiltonian cost on the variables we are using.

In [9]:
n_qubits = 8

qubit_labels = [
    "A004_LSB", "A004_MSB",
    "A047_LSB", "A047_MSB",
    "A026_LSB", "A026_MSB",
    "A017_LSB", "A017_MSB",
]

# Constant offset is irrelevant for QAOA optimization because it does not change
# which bitstring minimizes the objective.
c0 = 0.0

# Local fields h_i for H_C = sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j
h_terms = [
    (0, -0.048631),
    (1, -0.093696),
    (2, -0.036724),
    (3, -0.071445),
    (4, -0.070662),
    (5, -0.133323),
    (6, -0.112826),
    (7, -0.203428),
]

# Pair couplings J_ij
J_terms = [
    (0, 1, 0.003566),
    (0, 2, 0.001333),
    (0, 3, 0.002667),
    (0, 4, 0.002667),
    (0, 5, 0.005334),
    (0, 6, 0.004444),
    (0, 7, 0.008889),
    (1, 2, 0.002667),
    (1, 3, 0.005334),
    (1, 4, 0.005334),
    (1, 5, 0.010667),
    (1, 6, 0.008889),
    (1, 7, 0.017778),
    (2, 3, 0.002003),
    (2, 4, 0.002000),
    (2, 5, 0.004000),
    (2, 6, 0.003333),
    (2, 7, 0.006667),
    (3, 4, 0.004000),
    (3, 5, 0.008000),
    (3, 6, 0.006667),
    (3, 7, 0.013334),
    (4, 5, 0.008002),
    (4, 6, 0.006667),
    (4, 7, 0.013334),
    (5, 6, 0.013334),
    (5, 7, 0.026667),
    (6, 7, 0.022225),
]

ground_truth_bitstring = "11011110"

**Classical Icing Energy**

In [31]:
def ising_energy_from_x_list(x_list, h, J, ising_offset=0.0):
    # Use the convention x = (1 + s)/2  ->  s = 2x - 1
    s = np.array([2 * int(bit) - 1 for bit in x_list], dtype=float)

    energy = ising_offset

    for i in range(len(h)):
        energy += h[i] * s[i]

    for i in range(J.shape[0]):
        for j in range(i + 1, J.shape[1]):
            energy += J[i, j] * s[i] * s[j]

    return float(energy)

**Temporal checks**

In [11]:
print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth Ising energy:", ising_energy(ground_truth_bitstring, h_terms, J_terms, c0))

test_bitstrings = [
    "00000000",
    "11111111",
    "10101010",
    "01010101",
    ground_truth_bitstring,
]

for bs in test_bitstrings:
    print(bs, "->", ising_energy(bs, h_terms, J_terms, c0))

Ground-truth bitstring: 11011110
Ground-truth Ising energy: 0.27510699999999993
00000000 -> -0.5509330000000003
11111111 -> 0.9905369999999996
10101010 -> -0.24840300000000004
01010101 -> 0.21769499999999994
11011110 -> 0.27510699999999993


**Decode 2 bits per asset**

In [12]:
def decode_lots_from_bitstring(bitstring):
    """
    Decode 8 qubits into 4 asset lot values using:
    lot = LSB + 2 * MSB
    """
    bits = [int(b) for b in bitstring]
    lots = []

    for k in range(0, len(bits), 2):
        lsb = bits[k]
        msb = bits[k + 1]
        lot = lsb + 2 * msb
        lots.append(lot)

    return lots

**Verify the portfolio**

In [13]:
decoded_lots = decode_lots_from_bitstring(ground_truth_bitstring)
print("Decoded lots from ground-truth bitstring:", decoded_lots)

Decoded lots from ground-truth bitstring: [3, 2, 3, 1]


**Build the QAOA algorithm on Bloqade**


In [16]:
def build_qaoa_ising_kernel(n_qubits, h_terms, J_terms):
    @qasm2.extended
    def kernel(gamma: ilist.IList[float, Any], beta: ilist.IList[float, Any]):
        q = qasm2.qreg(n_qubits)

        # Initialize the register in the |+> state
        for i in range(n_qubits):
            qasm2.h(q[i])

        # Apply p QAOA layers
        for layer in range(len(gamma)):
            g = gamma[layer]
            b = beta[layer]

            # Single-qubit Z terms
            for k in range(len(h_terms)):
                term = h_terms[k]
                i = term[0]
                hi = term[1]
                qasm2.rz(q[i], 2.0 * g * hi)

            # Two-qubit ZZ terms
            for k in range(len(J_terms)):
                term = J_terms[k]
                i = term[0]
                j = term[1]
                Jij = term[2]

                qasm2.cx(q[i], q[j])
                qasm2.rz(q[j], 2.0 * g * Jij)
                qasm2.cx(q[i], q[j])

            # Mixer layer
            for i in range(n_qubits):
                qasm2.rx(q[i], 2.0 * b)

        return q

    return kernel

**Create the Kernel**

In [17]:
kernel = build_qaoa_ising_kernel(n_qubits, h_terms, J_terms)
print(kernel)

Method("kernel")


In [23]:
print(kernel)
print(type(kernel))

Method("kernel")
<class 'kirin.ir.method.Method'>


**Choose initial QAOA parameters**

In [ ]:
# P stands for the depth of the circuit, how many times we repeat the gates
p = 1

gamma_init = [0.3] * p
beta_init = [0.2] * p

**wrap in main and emit QASM**

In [19]:
@qasm2.extended
def main():
    return kernel(gamma_init, beta_init)

**print the generated circuit**

In [20]:
target = qasm2.emit.QASM2()
ast = target.emit(main)
qasm2.parse.pprint(ast)

OPENQASM 2.0;
include "qelib1.inc";
qreg q[8];
h q[0];
h q[1];
h q[2];
h q[3];
h q[4];
h q[5];
h q[6];
h q[7];
rz (-0.0291786) q[0];
rz (-0.0562176) q[1];
rz (-0.0220344) q[2];
rz (-0.042866999999999995) q[3];
rz (-0.0423972) q[4];
rz (-0.07999379999999999) q[5];
rz (-0.0676956) q[6];
rz (-0.1220568) q[7];
CX q[0], q[1];
rz (0.0021396) q[1];
CX q[0], q[1];
CX q[0], q[2];
rz (0.0007997999999999999) q[2];
CX q[0], q[2];
CX q[0], q[3];
rz (0.0016002) q[3];
CX q[0], q[3];
CX q[0], q[4];
rz (0.0016002) q[4];
CX q[0], q[4];
CX q[0], q[5];
rz (0.0032004) q[5];
CX q[0], q[5];
CX q[0], q[6];
rz (0.0026663999999999998) q[6];
CX q[0], q[6];
CX q[0], q[7];
rz (0.005333399999999999) q[7];
CX q[0], q[7];
CX q[1], q[2];
rz (0.0016002) q[2];
CX q[1], q[2];
CX q[1], q[3];
rz (0.0032004) q[3];
CX q[1], q[3];
CX q[1], q[4];
rz (0.0032004) q[4];
CX q[1], q[4];
CX q[1], q[5];
rz (0.006400199999999999) q[5];
CX q[1], q[5];
CX q[1], q[6];
rz (0.005333399999999999) q[6];
CX q[1], q[6];
CX q[1], q[7];
rz (0.01

**Brute-force classical check over all 256 bitstrings**

In [21]:
def all_bitstrings(n):
    return [format(x, f"0{n}b") for x in range(2**n)]

results = []
for bs in all_bitstrings(n_qubits):
    e = ising_energy(bs, h_terms, J_terms, c0)
    results.append((bs, e, decode_lots_from_bitstring(bs)))

results = sorted(results, key=lambda x: x[1])

print("Top 15 lowest-energy bitstrings from the Ising Hamiltonian:")
for rank, (bs, e, lots) in enumerate(results[:15], start=1):
    print(f"{rank:2d}. {bs}  energy={e:.8f}  lots={lots}")

Top 15 lowest-energy bitstrings from the Ising Hamiltonian:
 1. 00000000  energy=-0.55093300  lots=[0, 0, 0, 0]
 2. 00100000  energy=-0.52149100  lots=[0, 1, 0, 0]
 3. 10000000  energy=-0.51147100  lots=[1, 0, 0, 0]
 4. 00001000  energy=-0.49361700  lots=[0, 0, 1, 0]
 5. 00010000  energy=-0.49205300  lots=[0, 2, 0, 0]
 6. 10100000  energy=-0.47669700  lots=[1, 1, 0, 0]
 7. 01000000  energy=-0.47201100  lots=[2, 0, 0, 0]
 8. 00000010  energy=-0.45639900  lots=[0, 0, 0, 1]
 9. 00101000  energy=-0.45617500  lots=[0, 1, 1, 0]
10. 00110000  energy=-0.45459900  lots=[0, 3, 0, 0]
11. 10001000  energy=-0.44348700  lots=[1, 0, 1, 0]
12. 10010000  energy=-0.44192300  lots=[1, 2, 0, 0]
13. 00000100  energy=-0.43629500  lots=[0, 0, 2, 0]
14. 01100000  energy=-0.43190100  lots=[2, 1, 0, 0]
15. 00011000  energy=-0.41873700  lots=[0, 2, 1, 0]


**Check if your solution is near the minimum**

In [22]:
gt_energy = ising_energy(ground_truth_bitstring, h_terms, J_terms, c0)

rank = None
for idx, (bs, e, lots) in enumerate(results):
    if bs == ground_truth_bitstring:
        rank = idx + 1
        break

print("Ground-truth bitstring:", ground_truth_bitstring)
print("Ground-truth lots:", decode_lots_from_bitstring(ground_truth_bitstring))
print("Ground-truth energy:", gt_energy)
print("Ground-truth rank in Ising brute force:", rank)

Ground-truth bitstring: 11011110
Ground-truth lots: [3, 2, 3, 1]
Ground-truth energy: 0.27510699999999993
Ground-truth rank in Ising brute force: 207


**First diagnostic after debug**

In [25]:
gt_energy = ising_energy(ground_truth_bitstring, h_terms, J_terms, c0)

results_min = []
results_max = []

for bs in all_bitstrings(n_qubits):
    e = ising_energy(bs, h_terms, J_terms, c0)
    results_min.append((bs, e, decode_lots_from_bitstring(bs)))
    results_max.append((bs, -e, decode_lots_from_bitstring(bs)))

results_min = sorted(results_min, key=lambda x: x[1])
results_max = sorted(results_max, key=lambda x: x[1])

rank_min = None
rank_max = None

for idx, (bs, e, lots) in enumerate(results_min):
    if bs == ground_truth_bitstring:
        rank_min = idx + 1
        break

for idx, (bs, e, lots) in enumerate(results_max):
    if bs == ground_truth_bitstring:
        rank_max = idx + 1
        break

print("Ground-truth bitstring:", ground_truth_bitstring)
print("Rank if we minimize H:", rank_min)
print("Rank if we minimize -H:", rank_max)

Ground-truth bitstring: 11011110
Rank if we minimize H: 207
Rank if we minimize -H: 50


**Diagnostic 2**

In [26]:
reversed_gt = ground_truth_bitstring[::-1]

rank_reversed = None
for idx, (bs, e, lots) in enumerate(results_min):
    if bs == reversed_gt:
        rank_reversed = idx + 1
        break

print("Original ground-truth:", ground_truth_bitstring)
print("Reversed ground-truth:", reversed_gt)
print("Rank of reversed bitstring:", rank_reversed)
print("Decoded lots from reversed bitstring:", decode_lots_from_bitstring(reversed_gt))

Original ground-truth: 11011110
Reversed ground-truth: 01111011
Rank of reversed bitstring: 231
Decoded lots from reversed bitstring: [2, 3, 1, 3]


**Diagnostic 3**

In [27]:
flipped_gt = "".join("1" if b == "0" else "0" for b in ground_truth_bitstring)

rank_flipped = None
for idx, (bs, e, lots) in enumerate(results_min):
    if bs == flipped_gt:
        rank_flipped = idx + 1
        break

print("Original ground-truth:", ground_truth_bitstring)
print("Flipped ground-truth:", flipped_gt)
print("Rank of flipped bitstring:", rank_flipped)
print("Decoded lots from flipped bitstring:", decode_lots_from_bitstring(flipped_gt))

Original ground-truth: 11011110
Flipped ground-truth: 00100001
Rank of flipped bitstring: 44
Decoded lots from flipped bitstring: [0, 1, 0, 2]


**Diagnostic 4**

In [28]:
rev_flip_gt = "".join("1" if b == "0" else "0" for b in ground_truth_bitstring[::-1])

rank_rev_flip = None
for idx, (bs, e, lots) in enumerate(results_min):
    if bs == rev_flip_gt:
        rank_rev_flip = idx + 1
        break

print("Reverse+flip ground-truth:", rev_flip_gt)
print("Rank of reverse+flip bitstring:", rank_rev_flip)
print("Decoded lots from reverse+flip:", decode_lots_from_bitstring(rev_flip_gt))

Reverse+flip ground-truth: 10000100
Rank of reverse+flip bitstring: 24
Decoded lots from reverse+flip: [1, 0, 2, 0]


**Compare QUBO and Ising on the same solutions (These are still diagnostics)**